# Universal GraphRAG Pipeline

This notebook demonstrates how the same scoring algorithms and the same `GraphRAGPipeline` class work with **any graph dataset** — not just the Wikipedia Vote Network.

Two examples are shown side by side:

| | This project | Friend's notebook |
|---|---|---|
| **Dataset** | Wikipedia Vote Network (SNAP) | HotpotQA (HuggingFace) |
| **Graph type** | One large directed social graph | Many small undirected entity graphs |
| **Task** | Link prediction — which user trusts whom? | Context retrieval — which entity answers this question? |
| **Evaluation** | Precision@K | AUC-ROC + Average Precision |
| **Pipeline call** | Same `GraphRAGPipeline` | Same `GraphRAGPipeline` |

The algorithms (PPR, Common Neighbours, Jaccard, Adamic/Adar, Katz) are **identical** — only the dataset adapter changes.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from graphrag import EdgeListDataset, WikiVoteDataset, ContextGraphDataset, GraphRAGPipeline

---
## Part A — Wikipedia Vote Network (our project)

Task: **link_prediction**  
Metric: **Precision@10**

In [ ]:
# Use the WikiVoteDataset adapter — it handles SNAP text format OR pre-cleaned CSV
wiki_dataset = WikiVoteDataset(
    path=PROJECT_ROOT / "data" / "processed" / "graph_edges.csv",
    test_size=0.2,
    negatives_per_positive=10,
    seed=42,
)

print("Dataset summary:")
for k, v in wiki_dataset.summary().items():
    print(f"  {k:30s}: {v}")

In [ ]:
wiki_pipeline = GraphRAGPipeline(
    dataset=wiki_dataset,
    methods=["ppr", "cn", "jaccard", "aa", "katz"],   # short aliases work too
    k=10,
    verbose=True,
)

wiki_results = wiki_pipeline.run()
wiki_results

---
## Part B — Any other edge-list social network

Just point `EdgeListDataset` at any CSV with `source,target` columns.  
It does **not** need to be the Wikipedia dataset.

In [ ]:
# Example: drop in any other network CSV
# my_dataset = EdgeListDataset(
#     path="path/to/my_network.csv",
#     directed=True,          # or False for undirected
#     source_col="from",      # rename if your columns are different
#     target_col="to",
#     test_size=0.2,
#     negatives_per_positive=10,
#     seed=42,
# )
#
# pipeline = GraphRAGPipeline(my_dataset, methods=["ppr", "cn", "jaccard"])
# results  = pipeline.run()

# Also works directly with SNAP .txt files (comment lines starting with #):
# snap_dataset = EdgeListDataset(
#     path="data/raw/wiki-Vote.txt",
#     comment="#",
#     directed=True,
# )

print("Swap the path above to run on any edge-list social network.")

---
## Part C — HotpotQA entity co-occurrence graphs (friend's dataset)

Task: **context_retrieval**  
Metric: **AUC-ROC + Average Precision**

This replicates the AT3_SocialNetwork.ipynb approach but runs through the
same `GraphRAGPipeline` interface.  
Requires: `pip install datasets spacy && python -m spacy download en_core_web_sm`

In [ ]:
# ── Step 1: install dependencies if needed ──────────────────────────────────
# !pip install datasets spacy -q
# !python -m spacy download en_core_web_sm -q

try:
    import spacy
    from datasets import load_dataset
    HOTPOTQA_AVAILABLE = True
    print("HotpotQA dependencies available.")
except ImportError:
    HOTPOTQA_AVAILABLE = False
    print("Run: pip install datasets spacy && python -m spacy download en_core_web_sm")

In [ ]:
if HOTPOTQA_AVAILABLE:
    import networkx as nx
    nlp = spacy.load("en_core_web_sm")

    # ── Helper: build entity co-occurrence graph from one HotpotQA example ────
    ENTITY_LABELS = {"PERSON", "ORG", "GPE", "LOC", "WORK_OF_ART", "EVENT"}

    def extract_entities(text: str) -> list[str]:
        doc = nlp(text)
        return [e.text.lower().strip() for e in doc.ents if e.label_ in ENTITY_LABELS]

    def build_example_graph(example: dict) -> nx.Graph:
        G = nx.Graph()
        for _, sentences in zip(example["context"]["title"], example["context"]["sentences"]):
            for sentence in sentences:
                ents = list(set(extract_entities(sentence)))
                for i in range(len(ents)):
                    for j in range(i + 1, len(ents)):
                        u, v = ents[i], ents[j]
                        if G.has_edge(u, v):
                            G[u][v]["weight"] += 1
                        else:
                            G.add_edge(u, v, weight=1)
        return G

    # ── Step 2: load dataset and build records ─────────────────────────────────
    N_EXAMPLES = 100  # reduce to 50 for a quick test
    raw_dataset = load_dataset("hotpot_qa", "distractor", split=f"validation[:{N_EXAMPLES}]")

    records = []
    for example in raw_dataset:
        G = build_example_graph(example)
        if G.number_of_nodes() < 5:
            continue

        seeds    = extract_entities(example["question"])
        positives = set(extract_entities(example["answer"]))
        for title in example["supporting_facts"]["title"]:
            positives.update(extract_entities(title))

        if seeds and positives:
            records.append({"graph": G, "seeds": seeds, "positives": positives})

    print(f"Built {len(records)} valid records from {N_EXAMPLES} HotpotQA examples.")

In [ ]:
if HOTPOTQA_AVAILABLE and records:
    # ── Step 3: wrap in ContextGraphDataset and run the SAME pipeline ──────────
    hotpot_dataset = ContextGraphDataset(records, dataset_name="HotpotQA")

    print("Dataset summary:")
    for k, v in hotpot_dataset.summary().items():
        print(f"  {k:30s}: {v}")
    print()

In [ ]:
if HOTPOTQA_AVAILABLE and records:
    hotpot_pipeline = GraphRAGPipeline(
        dataset=hotpot_dataset,
        methods=["ppr", "jaccard", "aa", "katz"],  # same methods, different data
        verbose=True,
    )

    hotpot_results = hotpot_pipeline.run()
    hotpot_results

---
## Part D — Combine results: compare both datasets side by side

The algorithms mean the same thing on both datasets:
- **PPR wins on both** — it follows multi-hop paths, which is the core GraphRAG idea
- **Jaccard scores lowest on both** — it discounts hub nodes, which are meaningful connectors

In [ ]:
import pandas as pd

METHOD_LABELS = {
    "personalized_pagerank": "Personalised PageRank",
    "common_neighbors":      "Common Neighbours",
    "adamic_adar":           "Adamic / Adar",
    "jaccard":               "Jaccard",
    "katz":                  "Katz",
}

# Wikipedia Vote (link prediction)
wiki_display = wiki_results[["method", "mean_precision_at_k"]].copy()
wiki_display["method"] = wiki_display["method"].map(lambda m: METHOD_LABELS.get(m, m))
wiki_display = wiki_display.rename(columns={"mean_precision_at_k": "WikiVote P@10"})
wiki_display = wiki_display.set_index("method")

print("=== Wikipedia Vote Network (Precision@10) ===")
print(wiki_display.to_string())
print()

if HOTPOTQA_AVAILABLE and records:
    hotpot_display = hotpot_results[["method", "auc_roc", "average_precision"]].copy()
    hotpot_display["method"] = hotpot_display["method"].map(lambda m: METHOD_LABELS.get(m, m))
    hotpot_display = hotpot_display.set_index("method")

    print("=== HotpotQA (AUC-ROC / Average Precision) ===")
    print(hotpot_display.to_string())

---
## How to add your own dataset

### Option A — Edge list (any social / trust / citation network)

```python
from graphrag import EdgeListDataset, GraphRAGPipeline

dataset  = EdgeListDataset("my_network.csv", directed=True)
pipeline = GraphRAGPipeline(dataset, methods=["ppr", "cn", "jaccard"])
results  = pipeline.run()   # → Precision@K
```

### Option B — Per-example graphs (QA, knowledge graph, citation context)

```python
from graphrag import ContextGraphDataset, GraphRAGPipeline

records = [
    {"graph": G,           # networkx.Graph for this example
     "seeds": ["entity_a"],  # query / question entities
     "positives": {"entity_b", "entity_c"}},  # ground-truth relevant entities
    ...
]
dataset  = ContextGraphDataset(records, dataset_name="MyDataset")
pipeline = GraphRAGPipeline(dataset, methods=["ppr", "katz"])
results  = pipeline.run()   # → AUC-ROC + Average Precision
```

### Option C — Fully custom (subclass GraphDataset)

```python
from graphrag.dataset import GraphDataset
import networkx as nx
import pandas as pd

class MyDataset(GraphDataset):
    task = "link_prediction"   # or "context_retrieval"

    @property
    def name(self): return "My Custom Dataset"

    def get_graph(self):       return nx.DiGraph(...)  # your graph
    def get_candidates(self):  return pd.DataFrame(...)  # source, target, label

dataset  = MyDataset()
pipeline = GraphRAGPipeline(dataset, methods=["ppr", "jaccard"])
results  = pipeline.run()
```